In [ ]:
import polars as pl

df = pl.DataFrame({"city": ["NYC", "NYC", "LA"], "temp": [72, 74, 80]})
assert isinstance(df, pl.DataFrame)
assert df.shape == (3, 2)
assert df.schema["temp"] == pl.Int64

out = df.group_by("city").agg(pl.col("temp").mean().alias("mean_temp")).sort("city")
assert out["mean_temp"].to_list() == [80.0, 73.0]
print(out)

s = pl.Series("v", [1, 2, 3])
assert (s * 2).sum() == 12
print("DataFrame + Series eager ops OK")

In [ ]:
# Lazy engine: build a query plan, inspect it, collect via both engines.
import polars as pl
from polars.testing import assert_frame_equal

df = pl.DataFrame({"city": ["NYC", "NYC", "LA"], "temp_f": [72, 74, 80]})
lf = (
    df.lazy()
    .filter(pl.col("temp_f") > 72)
    .with_columns(temp_c=(pl.col("temp_f") - 32) * 5 / 9)
)
plan = lf.explain()
assert "FILTER" in plan
print(plan)

in_memory = lf.collect()
streaming = lf.collect(engine="streaming")
# Engines may differ by 1 ULP on float arithmetic — compare with tolerance.
assert_frame_equal(in_memory, streaming)
assert in_memory["temp_c"].round(1).to_list() == [23.3, 26.7]
print("lazy collect: in-memory == streaming engine")

In [ ]:
# Expression engine: window functions, conditionals, string namespace.
import polars as pl

d = pl.DataFrame({"k": ["a", "b", "a"], "v": [1, 2, 3], "name": ["foo", "Bar", "baz"]})
out = d.with_columns(
    total=pl.col("v").sum().over("k"),
    flag=pl.when(pl.col("v") > 1).then(pl.lit("hi")).otherwise(pl.lit("lo")),
    upper=pl.col("name").str.to_uppercase(),
)
assert out["total"].to_list() == [4, 2, 4]
assert out["flag"].to_list() == ["lo", "hi", "hi"]
assert out["upper"].to_list() == ["FOO", "BAR", "BAZ"]
print(out)
print("window + when/then + str namespace OK")

In [ ]:
# Round-trip: Parquet and CSV through in-memory buffers.
import io

import polars as pl
from polars.testing import assert_frame_equal

df = pl.DataFrame({"i": [1, 2, 3], "f": [1.5, 2.5, 3.5], "s": ["a", "b", "c"]})

pq = io.BytesIO()
df.write_parquet(pq)
pq.seek(0)
assert_frame_equal(df, pl.read_parquet(pq))

csv = io.BytesIO()
df.write_csv(csv)
csv.seek(0)
assert_frame_equal(df, pl.read_csv(csv))
print("Parquet + CSV round-trips preserve the frame exactly")

In [ ]:
import sys

import polars as pl
import polars._plr

active = sys.modules["polars._plr"].__name__
print(f"active runtime package: {active}")
assert active.startswith("_polars_runtime"), active
assert pl.thread_pool_size() >= 1

left = pl.DataFrame({"k": ["a", "b"], "x": [1, 2]})
right = pl.DataFrame({"k": ["a", "b"], "y": [10, 20]})
joined = left.join(right, on="k", how="inner").sort("k")
assert joined["y"].to_list() == [10, 20]

ctx = pl.SQLContext(frames={"t": joined})
res = ctx.execute("SELECT k, x + y AS s FROM t ORDER BY k").collect()
assert res["s"].to_list() == [11, 22]
print("join + SQL through the Rust engine OK")